# (Fully connected) Neural Networks

<div align="center">
  <img src="Medias/NeuralNetwork.png" width="500">
</div>

Given a Neural network with L number of layers, the following is true:

$$a^{(l)} = \sigma (W^{(l)} a^{(l-1)} + b^{(l)})$$
$$a^{(0)} = x^{(i)}$$

Here, $W^{(l)}$ represents a matrix containing the weighs at each layer, and $b^{(l)}$ is the biases of each neurons in layer $l$. More specifically, if in layer $l-1$ there are $m$ numbers of neurons, and in layer $l$ there are $n$ numbers of neurons, then $W^{(l)} \in \mathbb{R}^{n \times m}$. $\sigma$ here is the sigmoid function:

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

Just like ML models we've looked at in the past, we use a loss function to change these weighs and bias. Assuming $x^{(i)}$ are the features, $N_L$ is the number of output neurons, $f_{w,b}(x) := f(x)$ is our neural network, and $y^{(i)}$ is the actual correct data, then our loss function is given by:

$$\mathcal{L}(w,b) = \frac{1}{2}\sum_{n=1}^{N_L} (y^{(i)}_n - a^{(L)}_n)^2$$
$$a^{(L)} = f(x^{(i)})$$

Now we can introduce a way to update our weights and biases, known as Backpropagation. Lets first remind ourself of the simple update rule:

$$w \leftarrow w + \eta \cdot \nabla_w \mathcal{L}(w,b)$$
$$b \leftarrow b + \eta \cdot \nabla_b \mathcal{L}(w,b)$$

$W^{(L)}_{j,k}$ is updated thus with the following:


$$
\frac{\partial \mathcal{L}}{\partial W^{(L)}_{j,k}} = \frac{\partial \mathcal{L}}{\partial a^{(L)}_n} \cdot \frac{\partial a^{(L)}_n}{\partial z^{(L)}_n} \cdot \frac{\partial z^{(L)}_n}{\partial W^{(L)}_{j,k}}

$$

This simplifies to:

$$
\frac{\partial \mathcal{L}}{\partial W^{(L)}_{j,k}} = \underbrace{-(y^{(i)}_j - a^{(L)}_j) \cdot \sigma '(z^{(L)}_j)}_{\delta^{(L)}_j} \cdot a^{(L-1)}_k
$$

Here is an example using $W^{(L-1)}_{j,k}$:

$$
\frac{\partial \mathcal{L}}{\partial W^{(L-1)}_{j,k}} = \sum_{n=1}^{N_L} \left(\frac{\partial \mathcal{L}}{\partial a^{(L)}_n} \cdot \frac{\partial a^{(L)}_n}{\partial z^{(L)}_n} \cdot \frac{\partial z^{(L)}_n}{\partial a^{(L-1)}_j}\right) \cdot \frac{\partial a^{(L-1)}_j}{\partial z^{(L-1)}_j} \cdot \frac{\partial z^{(L-1)}_j}{\partial W^{(L-1)}_{j,k}}
$$

This quickly becomes impratical when we move deeper and deeper inside our neural network, and so we instead choose to represent the inner weights with:

$$
\frac{\partial \mathcal{L}}{\partial W^{(L-1)}_{j,k}} = \delta^{(L-1)}_j \cdot a^{(L-2)}_k
$$

where

$$\delta_j^{(l)} = \left( \sum_{m=1}^{N_{l+1}} \delta_m^{(l+1)} W_{m,j}^{(l+1)} \right) \cdot \sigma'(z_j^{(l)}), \text{ for }l\neq L$$
$$\delta_j^{(L)} = -(y^{(i)}_j - a^{(L)}_j) \cdot \sigma '(z^{(L)}_j)$$

Even with this however, computing the changes for each $w$ and $b$ becomes inefficient when we have hundres and thousands of indivitual connections. We instead can try to change each $W^{(l)}$ at once since computers are great at matrix operations.

$$W^{(l)} \leftarrow W^{(l)}  - \eta \frac{\partial \mathcal{L}}{\partial W^{(l)}}, \text{ where } \eta \in \mathbb{R}$$

where for any scalar function $f$,

$$\frac{\partial f}{\partial W} =
\begin{bmatrix}
\frac{\partial f}{\partial W_{1,1}} & \cdots & \frac{\partial f}{\partial W_{1,n}} \\
\vdots & \ddots & \vdots \\
\frac{\partial f}{\partial W_{m,1}} & \cdots & \frac{\partial f}{\partial W_{m,n}}
\end{bmatrix}$$

We can also redefine our Loss function in term of vectors and matrices.

$$\mathcal{L}(w,b) = \frac{1}{2} \left\|   y^{(i)}  - \sigma \left(W^{(L)} a^{(L-1)} + b^{(L)}\right)  \right\|^2$$

Remember that:

$$a^{(l)} = \sigma (W^{(l)} a^{(l-1)} + b^{(l)})$$
$$a^{(0)} = x^{(i)}$$


In general, we get these results:

$$\frac{\partial \mathcal{L}}{\partial W^{(L)}} = \delta^{(L)} ({a}^{(L-1)})^\top$$

$${\delta}^{(L)} = -({y} - {a}^{(L)}) \odot \sigma'({z}^{(L)})$$

and

$$\frac{\partial \mathcal{L}}{\partial {W}^{(l)}} = {\delta}^{(l)} ({a}^{(l-1)})^\top$$

$${\delta}^{(l)} = \left( ({W}^{(l+1)})^\top {\delta}^{(l+1)} \right) \odot \sigma'({z}^{(l)})$$

Below is a example of a neural network using Pytorch, which sought to classify images of a number. Each image is 28 by 28, and contain values that range from 0 to 255. Note that for every neuron that is not in layer $L$, it uses the ReLU activation defined as $f(x) = \max{(0,x)}$, and

$$f'(x) = 
\begin{cases} 
1 & \text{if } x > 0 \\
0 & \text{if } x \leq 0 
\end{cases}$$

In [45]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset
from torchvision import datasets, transforms

# 1. Hyperparameters
input_size = 28**2  # 28x28 pixels flattened; input layer
hidden_size1 = 256 #Hidden layer 1
hidden_size2 = 150 #Hidden layer 2
hidden_size3 = 75 #Hidden layer 3
num_classes = 10 #Output layer
learning_rate = 0.10 #\eta
batch_size = 1
epoch = 5

# 2. Load MNIST number dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
train_dataset_subset = Subset(train_dataset, list(range(30000)))
train_loader = torch.utils.data.DataLoader(dataset=train_dataset_subset, batch_size=batch_size, shuffle=True)

test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,batch_size=batch_size, shuffle=False)

# 3. Define the Architecture
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Linear layers represent the Wx + b we discussed
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)  
        self.fc3 = nn.Linear(hidden_size2, hidden_size3)  
        self.fc4 = nn.Linear(hidden_size3, num_classes)  
        self.sigmoid = nn.Sigmoid()
        self.relu = nn.ReLU()
    def forward(self, x):
        # x starts as (batch, 1, 28, 28), we need to flatten it to (batch, 784)
        x = x.view(-1, input_size)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.relu(x)
        x = self.fc4(x)
        x = self.sigmoid(x)
        return x #returns x as (batch, 10)

    def fit(self, epoch, train_loader):
        for epoch in range(epoch):
            for batch_idx, (data, targets) in enumerate(train_loader):
                criterion = nn.MSELoss()
                optimizer = optim.SGD(model.parameters(), lr=learning_rate)

                targets_onehot = torch.zeros(targets.size(0), num_classes)
                targets_onehot.scatter_(1, targets.unsqueeze(1), 1)

                outputs = self.forward(data)
                loss = criterion(outputs, targets_onehot)
                
                # Backwardpropagation
                optimizer.zero_grad() # Clear old gradients
                loss.backward()       # Compute dL/dW for all layers automatically
                optimizer.step()      # Update weights: W = W - lr * grad
                
                if batch_idx % 1000 == 0:
                    print(f"Batch {batch_idx}, Loss: {loss.item()}")

    def test(self, test_loader):
        self.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for data, targets in test_loader:
                outputs = model(data)
                preds = torch.argmax(outputs, dim=1)

                correct += (preds == targets).sum().item()
                total += targets.size(0)

        accuracy = correct / total
        print(f"Test Accuracy: {accuracy*100}%")
        return accuracy

model = SimpleNet()
model.fit(epoch=epoch, train_loader=train_loader)
model.test(test_loader)





Batch 0, Loss: 0.253286749124527
Batch 1000, Loss: 0.08439240604639053
Batch 2000, Loss: 0.08009611070156097
Batch 3000, Loss: 0.009037965908646584
Batch 4000, Loss: 0.004814493004232645
Batch 5000, Loss: 0.02282048389315605
Batch 6000, Loss: 0.010956620797514915
Batch 7000, Loss: 0.0017852643504738808
Batch 8000, Loss: 0.009335187263786793
Batch 9000, Loss: 0.034743063151836395
Batch 10000, Loss: 0.009408906102180481
Batch 11000, Loss: 7.504688142034865e-07
Batch 12000, Loss: 0.01878964528441429
Batch 13000, Loss: 0.0243295319378376
Batch 14000, Loss: 0.0007456255843862891
Batch 15000, Loss: 5.316716124070808e-05
Batch 16000, Loss: 9.607466199668124e-05
Batch 17000, Loss: 0.002996805589646101
Batch 18000, Loss: 0.005861709825694561
Batch 19000, Loss: 0.0069862171076238155
Batch 20000, Loss: 0.01408502645790577
Batch 21000, Loss: 3.835643838101532e-06
Batch 22000, Loss: 0.0064999498426914215
Batch 23000, Loss: 0.012079698964953423
Batch 24000, Loss: 0.0549522340297699
Batch 25000, Loss

0.9446

The problem with our previous formulation is that its very computationally heavy. The main way to compress our image yet still retain all essential data is through Convolutions Neural Network:

# Convolution Neural Network

<div align="center">
  <img src="Medias/CNN.png" width="800">
</div>

In Convolutional Neural Network (CNN), we first preprocess our image, $A \in \mathbb{R}^{28 \times 28}$. In CNN, there is the convolution itself and the Pooling process.

## Convolution

First, lets defined our Kernel, which is a $\mathbb{R}^{k \times k}$ matrix. The convolution includes:

$$\text{Conv}(M) = \sum_{i,j}\left(K \odot M\right)_{i,j} + b_i $$

We then pass this value into an activation function, typically ReLU. Note that sometimes the bias term is not included.

We repeat this convolution process by passing the Kernel through part of the image, starting from the top-left and moving one stide (the amount of pixels) over to the right each time. Each time we do, we compute the convolution of that part of A with K. In the case that we do not add padding, which is the addition of extra pixels outside the image to keep size consistency, then the dimensional process is summarized with the following:

$$\text{Conv}_{s}: \mathbb{R}^{n \times n} \to \mathbb{R}^{m \times m}$$

$$m = \left\lfloor \frac{n - k}{s} \right\rfloor + 1$$

Where $s$ is our stride length.

## Pooling

The Pool Size, just like the Kernel, is a $p \times p$ square that goes through the same process as the Kernel over the image, and has its own stride. Main difference is that:

$$\text{Pool}(M) = \max{(M)}$$

We can also use other choices of Pooling as well, like MinPooling and AveragePooling, and they work just as they sound.